# Nonlinear Optimization

**Modern Operations Research with Python**

While linear and integer programming cover many problems, many real-world OR applications involve **nonlinear** objectives or constraints (e.g., quadratic costs, economies of scale, convex/concave functions, geometric programming).

In this chapter we explore:
- Unconstrained and constrained nonlinear programming using **SciPy.optimize**
- Quadratic Programming (QP) examples
- Mixed-Integer Nonlinear Programming (MINLP) considerations
- When to use local vs global solvers

In [ ]:
import numpy as np
from scipy.optimize import minimize, Bounds, LinearConstraint, NonlinearConstraint
import matplotlib.pyplot as plt
from pulp import *

np.random.seed(42)
print("Libraries loaded: numpy, scipy.optimize, pulp")

## 5.1 Unconstrained Nonlinear Optimization

Classic example: Rosenbrock function (banana function) — a common test problem.

In [ ]:
def rosenbrock(x):
    return (1 - x[0])**2 + 100 * (x[1] - x[0]**2)**2

x0 = np.array([0.0, 0.0])
result = minimize(rosenbrock, x0, method='Nelder-Mead')

print("Optimal solution:", result.x)
print("Minimum value:", result.fun)
print("Success:", result.success)

## 5.2 Constrained Nonlinear Optimization (SciPy)

**Example**: Portfolio optimization with nonlinear risk (variance) and return constraints.

In [ ]:
# Simulated asset returns and covariance
n_assets = 4
returns = np.array([0.10, 0.15, 0.08, 0.12])
cov_matrix = np.array([
    [0.04, 0.02, 0.01, 0.015],
    [0.02, 0.09, 0.03, 0.04],
    [0.01, 0.03, 0.025, 0.02],
    [0.015, 0.04, 0.02, 0.06]
])

def portfolio_variance(w):
    return w.T @ cov_matrix @ w

def neg_return(w):
    return -w @ returns

# Constraints: sum weights = 1, weights >= 0, expected return >= 0.10
cons = (
    {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
    {'type': 'ineq', 'fun': lambda w: w @ returns - 0.10}
)

bounds = Bounds(0, 1)
w0 = np.ones(n_assets) / n_assets

result = minimize(portfolio_variance, w0, method='SLSQP', bounds=bounds, constraints=cons)

print("Optimal weights:", result.x)
print("Portfolio variance (risk):", result.fun)
print("Expected return:", result.x @ returns)

## 5.3 Quadratic Programming with PuLP (for linear constraints + quadratic objective)

Note: Standard PuLP is linear only. For true QP we often use SciPy or switch to CVXPY/Pyomo in advanced chapters.

In [ ]:
print("For full QP support beyond linear, consider CVXPY or Pyomo (covered in later chapters).")

## 5.4 Nonlinear Example: Economic Order Quantity (EOQ) with Nonlinear Holding Cost

Classic inventory model with nonlinear total cost.

In [ ]:
def eoq_total_cost(Q, D=1000, S=100, H=2):
    return (D / Q) * S + (Q / 2) * H   # classic EOQ

# Optimize EOQ
result_eoq = minimize(lambda Q: eoq_total_cost(Q[0]), [100], bounds=[(1, None)])
print("Optimal order quantity Q*:", result_eoq.x[0])
print("Minimum total cost:", result_eoq.fun)

## 5.5 Key Takeaways

- Use **SciPy.optimize.minimize** for general nonlinear problems.
- For convex quadratic problems with linear constraints, SciPy or CVXPY are excellent.
- True Mixed-Integer Nonlinear Programs (MINLP) require specialized solvers (Bonmin, Couenne, or commercial like Gurobi).
- Always check convexity for guarantees of global optimality.

**Exercises:**
1. Add a nonlinear constraint (e.g., maximum volatility) to the portfolio example.
2. Implement the geometric programming version of a production planning problem.
3. Compare local solver (SLSQP) vs global methods on a multimodal function.

In the next chapter we explore **Hybrid OR-ML Approaches**.